# 01.08 · First look at LangSmith tracing

**Module 01 · Models, Messages & Prompts** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 60–90 min |
| 📋 Prerequisites | Lessons 01.01–01.07 (especially 01.07: a trace is the run tree of a Runnable). |
| 🔑 Keys | `LANGSMITH_API_KEY` and `GROQ_API_KEY` (both optional; Parts 1–2 run offline) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-11 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (fake model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** runs, traces, projects and threads, and why you trace from day one.
2. **Capture** a run tree locally (no account needed) and read its types, names, timings, tags, metadata and tokens.
3. **Enable** LangSmith selectively with `tracing_context`, send traces to a named project, and flush them with `wait_for_all_tracers`.
4. **Trace your own Python functions** with `@traceable` so LangChain calls nest under them.
5. **Navigate** a trace in the LangSmith UI, and apply the privacy rules (what's in a trace, masking, retention).

## 📚 Official reading (verified 2026-10-11)

| Page | Read it for |
|---|---|
| [Observability concepts](https://docs.langchain.com/langsmith/observability-concepts) | **Main page.** Runs, traces, threads, projects, feedback, retention |
| [Trace LangChain applications](https://docs.langchain.com/langsmith/trace-with-langchain) | Env vars, selective tracing, projects, metadata/tags, run names, run IDs, flushing |
| [Tracing quickstart](https://docs.langchain.com/langsmith/observability-quickstart) · [Create an API key](https://docs.langchain.com/langsmith/create-account-api-key) | Account, key types (PAT vs service key), first trace, where to click in the UI |
| [Add metadata and tags](https://docs.langchain.com/langsmith/add-metadata-tags) | Tags vs metadata, and filtering by them in the UI |
| [Mask inputs and outputs](https://docs.langchain.com/langsmith/mask-inputs-outputs) | `LANGSMITH_HIDE_INPUTS` / `LANGSMITH_HIDE_OUTPUTS` and anonymizers |
| [LangSmith pricing](https://www.langchain.com/pricing) | Free Developer plan: **5k base traces/month** (verified 2026-10-11) |

---
## Step 0 · Get a LangSmith key (optional, 5 minutes, free)

1. Open [smith.langchain.com](https://smith.langchain.com) and sign up (GitHub/Google or email). The **Developer** plan is free; no card is needed for this course's volume.
2. Open [Settings → API Keys](https://smith.langchain.com/settings) → **Create API Key**. Choose a **Personal Access Token** (PAT: for personal scripts; *service keys* are for deployed apps), describe it as `mastery-laptop`, click **Create API Key** and copy it (it starts with `lsv2_`). It's shown **only once** ([create an API key](https://docs.langchain.com/langsmith/create-account-api-key)).
3. In Git Bash, open your `.env` (`code .env` or `notepad .env`) and set:
   ```
   LANGSMITH_API_KEY=lsv2_...your key...
   LANGSMITH_TRACING=false
   ```
   Leave `LANGSMITH_TRACING=false`: in this course we turn tracing **on per cell** (Part 3), so you decide what gets sent.
4. If your account is in the **EU** or another non-US region, also set `LANGSMITH_ENDPOINT` to your region's API URL from the [region table](https://docs.langchain.com/langsmith/trace-with-langchain#1-configure-your-environment). Otherwise authentication fails.
5. Restart the kernel (**Kernel → Restart Kernel**) so `.env` is reloaded.

**Checkpoint:** the setup cell below reports `LANGSMITH_API_KEY` as set (masked, as in Lesson 00.04), never the key itself.

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2"

**You should now see** nothing, or *"Note: you may need to restart the kernel to use updated packages."* Restart only if a package was **upgraded** after you had already imported it (Lesson 00.07). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.03).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-groq", "langsmith",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/01-models-messages-prompts/).
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import the course package (installed editable in Lesson 00.02; src/ fallback otherwise).
try:
    import mastery
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import mastery
    print("ℹ️ Loaded mastery from src/. Fix once: python -m pip install -e \".[dev]\"")
print("Repo root:", REPO_ROOT)

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY', 'LANGSMITH_TRACING', 'LANGSMITH_PROJECT')
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Lesson flag: LangSmith cells run only with a key, so Restart & Run All never crashes without one.
import os

HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
PROJECT = "m01-first-traces"   # CONVENTIONS: LangSmith projects are named mNN-<slug>.
print("HAVE_LANGSMITH_KEY:", HAVE_LANGSMITH_KEY, "| tracing env:", os.environ.get("LANGSMITH_TRACING", "unset"))

---
## Part 1 · Why trace, and what a trace is

Without tracing, a failing LLM app is a black box: *which* prompt was sent? What did the model return before the parser broke? How long did each step take? How many tokens? LangSmith records this for every call ([concepts](https://docs.langchain.com/langsmith/observability-concepts)):

| Term | Meaning | In Lesson 01.07 terms |
|---|---|---|
| **Run** (a span) | One unit of work: a prompt format, a model call, a parser, a tool | One step of a pipeline |
| **Trace** | All runs of one operation, as a tree, sharing a trace ID | One `chain.invoke(...)` |
| **Project** | A folder of traces | e.g. `m01-first-traces` |
| **Thread** | Several traces of one conversation (linked by `thread_id` metadata) | Module 03 |

LangChain creates these runs **already**, for its own callbacks. LangSmith just uploads them. So you can see the exact same tree **locally**, with no account, using `collect_runs`.

In [ ]:
# 🧪 Offline: capture the run tree of one pipeline call locally. Nothing leaves your machine.
from langchain.messages import AIMessage
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tracers.context import collect_runs

prompt = ChatPromptTemplate.from_messages([("system", "Explain terms in {style} style."), ("human", "Explain: {term}")])
usage = {"input_tokens": 24, "output_tokens": 12, "total_tokens": 36}
fake_model = GenericFakeChatModel(messages=iter([AIMessage(
    "A trace is the tree of runs for one operation.", usage_metadata=usage,
    response_metadata={"model_name": "openai/gpt-oss-20b"})]))
explain = (prompt | fake_model | StrOutputParser()).with_config(run_name="explain_term")

with collect_runs() as collector:
    explain.invoke({"style": "one-sentence", "term": "trace"},
                   config={"tags": ["m01", "lesson-01.08"], "metadata": {"user_id": "learner-1", "variant": "A"}})
root = collector.traced_runs[0]
print("root run:", root.name, "| trace_id:", root.trace_id)

In [ ]:
# 🧪 Offline: print the tree like LangSmith's Details view: type, name, latency, tags, tokens.
def run_tokens(run) -> dict | None:
    """Token usage of an LLM run, read from its serialized output message (None for other runs)."""
    try:
        return run.outputs["generations"][0][0]["message"]["kwargs"].get("usage_metadata")
    except (KeyError, IndexError, TypeError):
        return None


def print_tree(run, depth: int = 0) -> None:
    """Print a run and its children, indented."""
    ms = (run.end_time - run.start_time).total_seconds() * 1000
    tags = [t for t in run.tags if not t.startswith("seq:")]   # Hide LangChain's internal step tags.
    tokens = run_tokens(run)
    print(f"{'    ' * depth}{run.run_type:<7} {run.name:<22} {ms:6.1f} ms  tags={tags}"
          + (f"  tokens={tokens['total_tokens']}" if tokens else ""))
    for child in run.child_runs:
        print_tree(child, depth + 1)


print_tree(root)
print("\nmetadata (inherited by children):", root.extra["metadata"])
print("LLM run input:", collector.traced_runs[0].child_runs[1].inputs)

### What just happened?

```
chain   explain_term   ...
    prompt  ChatPromptTemplate
    llm     GenericFakeChatModel   tokens=36
    parser  StrOutputParser
```

- One `invoke` produced **one trace with four runs**: the root `chain` (named by `run_name`) and one child per pipeline step, each with its own type (`chain`, `prompt`, `llm`, `parser`), timing and inputs/outputs.
- **Tags and metadata from `config` were inherited** by every child, as the [docs note](https://docs.langchain.com/langsmith/trace-with-langchain#add-metadata-and-tags-to-traces). In LangSmith you'll filter by them (`variant = A`, `user_id = learner-1`).
- The `llm` run stores the **exact prompt sent** and the tokens used. That's the first thing you check when an answer is wrong.

---
## Part 2 · Naming things so traces are readable

Traces are only useful if you can find things in them. Three cheap habits:

| Habit | How | Why |
|---|---|---|
| Name the root run | `.with_config(run_name="explain_term")` | Otherwise every trace is called `RunnableSequence` |
| Tag the experiment | `config={"tags": ["m01", "variant-a"]}` | Filter and compare in the UI |
| Add metadata | `config={"metadata": {"user_id": ..., "prompt_version": "v2"}}` | Group by user/version; `thread_id` links conversations (Module 03) |

You can also set your own **run ID** (`config={"run_id": uuid7()}`) to find a trace later from your logs ([docs](https://docs.langchain.com/langsmith/trace-with-langchain#access-run-span-id-for-langchain-invocations)).

---
## Part 3 · Sending a real trace to LangSmith

**Selective tracing:** keep `LANGSMITH_TRACING=false` and wrap the calls you want in `langsmith.tracing_context(enabled=True, project_name=...)`. Tracing uploads happen in a **background thread**; in a notebook they finish on their own, but scripts and serverless functions can exit first. `wait_for_all_tracers()` blocks until everything is sent.

In [ ]:
# 💸 Calls Groq (fast model), ~300 tokens, and uploads one trace to LangSmith project "m01-first-traces".
import langsmith as ls
from langchain_core.tracers.langchain import wait_for_all_tracers
from langchain_core.utils.uuid import uuid7

from mastery.models import get_model

if not (HAVE_GROQ_KEY and HAVE_LANGSMITH_KEY):
    print("⏭️ Skipped: needs GROQ_API_KEY and LANGSMITH_API_KEY (see Step 0).")
else:
    llm = get_model("fast", temperature=0, max_tokens=300, reasoning_effort="low")
    live_chain = (prompt | llm | StrOutputParser()).with_config(run_name="explain_term")
    run_id = uuid7()
    with ls.tracing_context(enabled=True, project_name=PROJECT):
        answer = live_chain.invoke(
            {"style": "one-sentence", "term": "observability"},
            config={"run_id": run_id, "tags": ["m01", "lesson-01.08"], "metadata": {"variant": "A"}},
        )
    wait_for_all_tracers()   # Flush the background uploads before we look for the trace.
    print("answer:", answer)
    print("run_id:", run_id)

### What just happened? (with both keys)

- The cell answered and printed a `run_id`. Only the calls **inside** the `with` block were traced; the `m01-first-traces` project is created automatically on the first trace.
- Why not read the run back with code? The SDK's run-reading API is **changing right now**: as of `langsmith` 0.14.7, `client.read_run()` prints a deprecation warning (removal after 2027-01-31), and its replacement `client.runs.retrieve()` is **async** and needs the project's UUID ([migration guide](https://docs.langchain.com/langsmith/smithdb-sdk-migration-runs)). The UI is the stable way for now; Module 12 queries traces with the current SDK. (Another reason to read deprecation warnings: Lesson 00.05.)

**Now click through (5 minutes):**

1. Open [smith.langchain.com](https://smith.langchain.com) → **Tracing** → project **m01-first-traces** → the newest `explain_term` row. (In an agent-based workspace, select the agent in the top bar first, per the [quickstart](https://docs.langchain.com/langsmith/observability-quickstart#4-view-your-trace).) Use the search box with your `run_id` if there are many rows.
2. **Details** tab: the run tree from Part 1, with real timings. Click **ChatGroq**: see the exact system + human messages sent, the output, and the token count.
3. Find the **tags** (`m01`, `lesson-01.08`) and **metadata** (`variant: A`) on the root run.
4. Back in the project list, add a filter on tag `lesson-01.08` and note the latency and token columns.

---
## Part 4 · `@traceable`: your own functions in the tree

Real apps have Python code around model calls: loading data, cleaning text, post-processing. Decorate those functions with `@traceable` and every LangChain call inside nests under them as children.

In [ ]:
# 💸 Calls Groq twice (fast model), ~500 tokens. One trace: a parent function with two model calls inside.
from langsmith import traceable


@traceable(name="glossary_builder", run_type="chain")
def build_glossary(terms: list[str]) -> dict[str, str]:
    """Explain each term; the whole loop becomes one trace with one child per call."""
    return {term: live_chain.invoke({"style": "one-sentence", "term": term}) for term in terms}


if not (HAVE_GROQ_KEY and HAVE_LANGSMITH_KEY):
    print("⏭️ Skipped: needs GROQ_API_KEY and LANGSMITH_API_KEY (see Step 0).")
else:
    with ls.tracing_context(enabled=True, project_name=PROJECT, tags=["m01", "lesson-01.08"]):
        glossary = build_glossary(["span", "latency"])
    wait_for_all_tracers()
    for term, text in glossary.items():
        print(f"{term}: {text}")

**In LangSmith you should see** one `glossary_builder` trace with **two** `explain_term` children, each with its own `ChatGroq` run. The parent shows the total latency. This is how you'll trace agents and graphs later: the structure appears automatically.

---
## Part 5 · What's in a trace: privacy and cost

A trace contains **full prompts and outputs**: whatever users typed, documents you retrieved, model answers.

| Rule | How |
|---|---|
| Never put secrets in prompts or metadata | API keys, passwords, tokens don't belong in model input anyway |
| Hide sensitive inputs/outputs | `LANGSMITH_HIDE_INPUTS=true` / `LANGSMITH_HIDE_OUTPUTS=true`, or an anonymizer ([masking docs](https://docs.langchain.com/langsmith/mask-inputs-outputs)) |
| Know the retention | SaaS keeps traces **180 days** from ingestion, then deletes them ([concepts](https://docs.langchain.com/langsmith/observability-concepts)) |
| Watch the quota | Free plan: 5k base traces/month. One `invoke` = one trace, however many runs it has |
| Trace selectively in shared notebooks | `tracing_context(enabled=True)` around what you need |

---
## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| No traces appear | Call wasn't inside `tracing_context(enabled=True)` and `LANGSMITH_TRACING` is false | Wrap the call, or set the env var to `true` and restart the kernel |
| `401`/`403` from LangSmith | Wrong key, or EU/other-region account without `LANGSMITH_ENDPOINT` | Re-copy the key; set the endpoint for your region |
| Traces go to the wrong workspace | Key linked to several workspaces | Set `LANGSMITH_WORKSPACE_ID` ([docs](https://docs.langchain.com/langsmith/trace-with-langchain#1-configure-your-environment)) |
| Script ends and traces are missing | Process exited before the background upload | `wait_for_all_tracers()` in a `finally:` |
| Old tutorial uses `LANGCHAIN_TRACING_V2`, `LANGCHAIN_API_KEY` | Pre-`LANGSMITH_*` names | Use `LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT` |
| Every trace is called `RunnableSequence` | No `run_name` | `.with_config(run_name="...")` |

## ⚠️ Production notes

> ⚠️ **Production note: tracing in production.**
> Traces hold user data. In production: decide what may be logged (mask or hide PII), use separate projects per environment (`app-dev`, `app-prod`), sample high-volume traffic instead of tracing 100%, attach `user_id`/`thread_id`/`prompt_version` metadata for debugging, and flush tracers on shutdown. Module 12 adds evaluations and monitoring on top of these traces.

## 🏋️ Exercises

1. **(Easy)** In Part 1, add `.with_config(run_name="format_prompt")` to `prompt` inside the pipeline. Re-run the tree. Which line changed?
2. **(Easy)** With keys: run Part 3 twice with `metadata={"variant": "A"}` and `"B"` and two different `style`s. In LangSmith, filter by metadata `variant` and compare latency and tokens.
3. **(Medium)** Use `collect_runs` around `llm.batch([...3 prompts...])` (or the fake model offline). How many root runs do you get? What does that mean for your 5k traces/month quota?
4. **(Stretch)** Make a deliberately failing pipeline (a `RunnableLambda` that raises after the model). Trace it. Find the error in LangSmith: which run is red, and what does the root run show?

<details>
<summary><b>Show solutions</b></summary>

**1.** The `prompt` line shows `format_prompt` instead of `ChatPromptTemplate`.

**3.** Three root runs: **each input of a batch is its own trace** (3 traces of the quota).
```python
fake = GenericFakeChatModel(messages=iter([AIMessage("a"), AIMessage("b"), AIMessage("c")]))
with collect_runs() as c:
    fake.batch(["x", "y", "z"])
print(len(c.traced_runs))   # 3
```

**4.** The failing `RunnableLambda` run is marked as an error with the traceback; the root run is marked as an error too, while the earlier `llm` run succeeded. That's how you find *where* a pipeline broke.
</details>

## ✅ Checkpoint

- [ ] I can define run, trace, project and thread.
- [ ] I captured and printed a run tree locally with `collect_runs`.
- [ ] (With key) I sent a trace to `m01-first-traces` with tags, metadata and a `run_id`, and found it in the UI.
- [ ] (With key) I traced a `@traceable` function with nested model calls.
- [ ] I know what a trace contains, how to mask it, how long it's kept, and the free quota.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Module 03 will link conversation turns into **threads**. In the [observability concepts](https://docs.langchain.com/langsmith/observability-concepts) page, find which metadata key groups traces into a thread, and how threads differ from traces and trajectories (the comparison table). Log the answer and the link in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- LangChain already records a run tree for every call; LangSmith uploads and visualizes it.
- Trace selectively with `tracing_context`, name runs, tag experiments, and flush with `wait_for_all_tracers`.
- `@traceable` puts your own functions in the tree.
- Traces contain user data: mask, separate projects, mind retention and quota.
- Next: **Project 01**, which uses everything from this module.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: the **module project**: read [project/README.md](project/README.md) (spec, modes, acceptance criteria).

[Module 01 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)